In [ ]:
# Import libraries we will use in the analysis.
# NumPy is the numerical library;
# Pandas is the library to handle data sets;
# yFinance is the library that allows us to download financial data from Yahoo;
# DateTime is the library that allows us to create variables in date format (year, month, day, minutes, seconds);
# MatPlotLib is the library to construct graphs.

import numpy as np
import pandas as pd
import yfinance as yf
import datetime as dt
import matplotlib.pyplot as plt

In [ ]:
# Set starting and ending date for dowloading data from Yahoo Finance.
# The variable "start" contains the first date we would like to download data. We choose January 1, 2018.
# The variable "end" contains the last date we would like to download data. We choose May 15, 2026.
# If we would like to use the lastest available date we can use "dt.datetime.now()" instead.

start = dt.datetime(2018,1,1)
end = dt.datetime(2026,5,15)

In [ ]:
# Download data for Bitcoin and Ether. We also download data for S&P500 index and yields on 10 years Treasury Bonds.
# The S&P500 will be used to calculate the return from the market potfolio, r_m. The treasury yilds will be used
# to calculate the risk-free rate, r_f. The ticker's name for S&P500 is "^GSPC"; the ticker's name for treasury yields is "^TNX".
# We place the downloaded data for Bitcoin in dataframe "btc_data"; the downloaded data for Ether in dataframe "eth_data";
# the downloaded data for S&P500 in dataframe sp500_data; the downloaded data for tresury yields in dataframe tby_data.

btc_data = yf.download('BTC-USD', start, end, progress=False)
eth_data = yf.download('ETH-USD', start, end, progress=False)
sp500_data = yf.download('^GSPC', start, end, progress=False)
tby_data = yf.download('^TNX', start, end, progress=False)

In [ ]:
# Create a new dataframe with "Close" prices of Bitcoin, Ether, S&P500.
# We call the first column "btc_price", the second column "eth_price", the third column "sp500"

DataSet = pd.concat([btc_data['Close'],eth_data['Close'],sp500_data['Close']], axis=1)

In [ ]:
# We rename the variables so that it is clear that they contain prices

DataSet = DataSet.rename(columns={"BTC-USD": "BTC_price","ETH-USD": "ETH_price","^GSPC": "SP500_price"})

In [ ]:
# Since in S&P500 there are missing value (NaN) for non-trading days (holidays), we eliminate the rows
# with missing values (NaN)

DataSet=DataSet.dropna()

In [ ]:
# Compute and add to DataSet daily returns for Bitcoin, Ether and S&P500 using the logarithmic formula

DataSet['BTC_ret']=np.log(DataSet['BTC_price'])-np.log(DataSet['BTC_price'].shift(1))
DataSet['ETH_ret']=np.log(DataSet['ETH_price'])-np.log(DataSet['ETH_price'].shift(1))
DataSet['SP500_ret']=np.log(DataSet['SP500_price'])-np.log(DataSet['SP500_price'].shift(1))

In [ ]:
# Since the first return is missing, we eliminate the first rows with missing values (NaN)

DataSet=DataSet.dropna()

In [ ]:
# Compute means, standard deviations, covariances and correlations of daily returns for Bitcoin, Ether and S&P500.
# Then print the means, standard deviations and correlations.

Mean_ret=DataSet[['BTC_ret','ETH_ret','SP500_ret']].mean()
Std_ret=DataSet[['BTC_ret','ETH_ret','SP500_ret']].std()
Cov_ret=DataSet[['BTC_ret','ETH_ret','SP500_ret']].cov()
Corr_ret=DataSet[['BTC_ret','ETH_ret','SP500_ret']].corr()

print("Means")
print(Mean_ret)
print("")
print("Standard Deviations")
print(Std_ret)
print("")
print("Correlations")
print(Corr_ret)

In [ ]:
# Define expected market return, r_m

r_m=Mean_ret['SP500_ret']

In [ ]:
# Define risk-free return, r_f
# Extract "Close" yields in treasury bonds and transform in daily returns.

tby_daily=(1+tby_data['Close']/100)**(1/365)-1
r_f=tby_daily.mean()
r_f=r_f.item()

In [ ]:
#Compute the betas for each investment.

beta_vec=Cov_ret['SP500_ret']/(Std_ret['SP500_ret']**2)

In [ ]:
#Given the betas, we compute the expected returns according to CAPM

CAPM_ret=r_f+beta_vec*(r_m-r_f)

In [ ]:
#Place the returns in a new dataframe and convert in yearly returns

Return_Mx=pd.concat([CAPM_ret, Mean_ret],axis=1,keys=['CAPM','Empirical'])

Return_Mx*365*100